# 03 JUNE 2026

In [ ]:
import pandas as pd
import re
import emoji
from nltk import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import tensorflow as tf
import torch
from torch import nn,optim

path = r"C:\Users\nisha\OneDrive\Desktop\Nisha\GUVI\MiniProject\CommentToxicity\Data\train.csv"
sw = stopwords.words('english')

df_train = pd.read_csv(path)
df_train.drop(['id'], axis=1,inplace=True)

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|https\S+|www\S+','',text)
    text = re.sub(r"<.*?>",'',text)
    text = re.sub(r'@\w+|#\w+','',text)
    text = re.sub(r'(.)\1{2,}',r'\1\1',text)
    text = re.findall(r'[a-zA-Z]|\s',text)
    text = emoji.replace_emoji(text,'')
    text = re.sub(r'\s+',' ',text)
    tokens = word_tokenize(text)
    tokens = [token for token in tokens if token not in sw]
    lemmatizer = WordNetLemmatizer()
    tokens = [lemmatizer.lemmatize(token) for token in tokens]
    return " ".join(tokens)

df_train['cleaned_comment'] = df_train['comment_text'].apply(clean_text)
X = df_train['cleaned_comment']
df_train['toxicity_ind'] = df_train[['toxic',
                         'severe_toxic',
                         'obscene',
                         'threat',
                         'insult',
                         'identity_hate']].sum(axis=1)
df_train['toxicity_ind'] = df_train['toxicity_ind'].apply(lambda x: 1 if x > 0 else 0)

vectorizer = tf.keras.layers.TextVectorization(
    max_tokens = 10000,
    output_sequence_length =200,
    output_mode = 'int' 
)
vectorizer.adapt(X.values)
vectorized_text = vectorizer(X.values)
vocab_size = len(vectorizer.get_vocabulary())
#print("Vocabulary Size:", vocab_size)

class SimpleRNN(nn.Module):
    def __init__(self,vocab_size,embed_size,hidden_size,output_size):
        super(SimpleRNN,self).__init__()
        self.embedding = nn.Embedding(vocab_size,embedding_dim=embed_size)
        self.rnn = nn.RNN(embed_size,hidden_size,batch_first=True)
        self.fc = nn.Linear(hidden_size,output_size)
        
    def forward(self,X):
        X = self.embedding(X)
        out,ht =self.rnn(X)
        out = out[:,-1,:]
        out = self.fc(out)
        return out
        
X = torch.tensor(vectorized_text,dtype=torch.long)
y = torch.tensor(df_train['toxicity_ind'].values,dtype=torch.long)

model = SimpleRNN(vocab_size, embed_size=8, hidden_size=16, output_size=2)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)
for epoch in range(101):
    output = model(X)
    loss = criterion(output, y)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if epoch % 10 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.4f}")


Epoch 0, Loss: 0.6778
Epoch 10, Loss: 0.3453
Epoch 20, Loss: 0.3292


In [2]:
!pip3 install scikit-learn

  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
   ---------------------------------------- 0.0/8.2 MB ? eta -:--:--
   --------------------- ------------------ 4.5/8.2 MB 24.4 MB/s eta 0:00:01
   ---------------------------------------- 8.2/8.2 MB 22.1 MB/s eta 0:00:00
   ---------------------------------------- 0.0/36.5 MB ? eta -:--:--
   ----- ---------------------------------- 5.0/36.5 MB 23.2 MB/s eta 0:00:02
   ------------ --------------------------- 11.5/36.5 MB 27.8 MB/s eta 0:00:01
   ------------------ --------------------- 17.3/36.5 MB 28.0 MB/s eta 0:00:01
   ------------------------ --------------- 22.3/36.5 MB 27.1 MB/s eta 0:00:01
   ------------------------------ --------- 28.0/36.5 MB 27.0 MB/s eta 0:00:01
   ------------------------------------- -- 34.6/36.5 MB 27.5 MB/s eta 0:00:01
   ---------------------------------------- 36.5/36.5 MB 25.0 MB/s eta 0:00:00
Using cached threadpoolctl-3.6.0-py3-none-any.whl (18 kB)



[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import pandas as pd
import re
import emoji
from nltk import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import tensorflow as tf
import torch
from torch import nn,optim
from sklearn.model_selection import train_test_split
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import classification_report

path = r"C:\Users\nisha\OneDrive\Desktop\Nisha\GUVI\MiniProject\CommentToxicity\Data\train.csv"

df_train = pd.read_csv(path)
df_train.drop(['id'], axis=1,inplace=True)

lemmatizer = WordNetLemmatizer()
sw = stopwords.words('english')
    
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|https\S+|www\S+','',text)
    text = re.sub(r"<.*?>",'',text)
    text = re.sub(r'@\w+|#\w+','',text)
    text = re.sub(r'(.)\1{2,}',r'\1\1',text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = emoji.replace_emoji(text, '')
    text = re.sub(r'\s+', ' ', text).strip()
    tokens = word_tokenize(text)
    tokens = [token for token in tokens if token not in sw]
    tokens = [lemmatizer.lemmatize(token) for token in tokens]
    return " ".join(tokens)

df_train['cleaned_comment'] = df_train['comment_text'].apply(clean_text)
X = df_train['cleaned_comment']
df_train['toxicity_ind'] = df_train[['toxic',
                         'severe_toxic',
                         'obscene',
                         'threat',
                         'insult',
                         'identity_hate']].sum(axis=1)
df_train['toxicity_ind'] = df_train['toxicity_ind'].apply(lambda x: 1 if x > 0 else 0)

vectorizer = tf.keras.layers.TextVectorization(
    max_tokens = 30000,
    output_sequence_length =100,
    output_mode = 'int' 
)
vectorizer.adapt(X.values)
vectorized_text = vectorizer(X.values)
vocab_size = len(vectorizer.get_vocabulary())
#print("Vocabulary Size:", vocab_size)

class SimpleRNN(nn.Module):
    def __init__(self,vocab_size,embed_size,hidden_size,output_size):
        super(SimpleRNN,self).__init__()
        self.embedding = nn.Embedding(vocab_size,embedding_dim=embed_size)
        self.rnn = nn.RNN(embed_size,hidden_size,batch_first=True)
        self.fc = nn.Linear(hidden_size,output_size)
        
    def forward(self,X):
        X = self.embedding(X)
        out,ht =self.rnn(X)
        out = out[:,-1,:]
        out = self.fc(out)
        return out
        
#X = torch.tensor(vectorized_text.numpy(),dtype=torch.long)
#y = torch.tensor(df_train['toxicity_ind'].values,dtype=torch.float32).reshape(-1,1)

X_train, X_val, y_train, y_val = train_test_split(
    vectorized_text.numpy(),
    df_train['toxicity_ind'].values,
    test_size=0.2,
    random_state=42
)

X_train = torch.tensor(X_train, dtype=torch.long)
X_val = torch.tensor(X_val, dtype=torch.long)

y_train = torch.tensor(y_train, dtype=torch.float32).reshape(-1,1)
y_val = torch.tensor(y_val, dtype=torch.float32).reshape(-1,1)

train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(
    train_dataset,
    batch_size=128,
    shuffle=True
)

val_dataset = TensorDataset(X_val, y_val)
val_loader = DataLoader(
    val_dataset,
    batch_size=128
)

model = SimpleRNN(vocab_size, embed_size=32, hidden_size=64, output_size=1)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
for epoch in range(10):
    model.train()
    for batch_X, batch_y in train_loader:
        output = model(batch_X)
        loss = criterion(output, batch_y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    print(f"Epoch {epoch}, Loss: {loss.item():.4f}")

all_preds = []
all_labels = []
model.eval()
with torch.no_grad():
    for batch_X, batch_y in val_loader:
        logits = model(batch_X)
        probs = torch.sigmoid(logits)
        preds = (probs > 0.5).float()
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(batch_y.cpu().numpy())
print(classification_report(
        all_labels,
        all_preds
    )
)

Epoch 0, Loss: 0.2178
Epoch 1, Loss: 0.4985
Epoch 2, Loss: 0.3790
Epoch 3, Loss: 0.2726
Epoch 4, Loss: 0.2714
Epoch 5, Loss: 0.6444
Epoch 6, Loss: 0.5477
Epoch 7, Loss: 0.3159
Epoch 8, Loss: 0.2008
Epoch 9, Loss: 0.3223
              precision    recall  f1-score   support

         0.0       0.90      1.00      0.95     28671
         1.0       0.45      0.01      0.03      3244

    accuracy                           0.90     31915
   macro avg       0.68      0.51      0.49     31915
weighted avg       0.85      0.90      0.85     31915



In [2]:
import pandas as pd
import numpy as np
import re
import emoji
from nltk import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import tensorflow as tf
import torch
from torch import nn,optim
from sklearn.model_selection import train_test_split
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix

path = r"C:\Users\nisha\OneDrive\Desktop\Nisha\GUVI\MiniProject\CommentToxicity\Data\train.csv"

df_train = pd.read_csv(path)
df_train.drop(['id'], axis=1,inplace=True)

lemmatizer = WordNetLemmatizer()
sw = stopwords.words('english')
    
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|https\S+|www\S+','',text)
    text = re.sub(r"<.*?>",'',text)
    text = re.sub(r'@\w+|#\w+','',text)
    text = re.sub(r'(.)\1{2,}',r'\1\1',text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = emoji.replace_emoji(text, '')
    text = re.sub(r'\s+', ' ', text).strip()
    tokens = word_tokenize(text)
    tokens = [token for token in tokens if token not in sw]
    tokens = [lemmatizer.lemmatize(token) for token in tokens]
    return " ".join(tokens)

df_train['cleaned_comment'] = df_train['comment_text'].apply(clean_text)
X = df_train['cleaned_comment']
df_train['toxicity_ind'] = df_train[['toxic',
                         'severe_toxic',
                         'obscene',
                         'threat',
                         'insult',
                         'identity_hate']].sum(axis=1)
df_train['toxicity_ind'] = df_train['toxicity_ind'].apply(lambda x: 1 if x > 0 else 0)

vectorizer = tf.keras.layers.TextVectorization(
    max_tokens = 30000,
    output_sequence_length =100,
    output_mode = 'int' 
)
vectorizer.adapt(X.values)
vectorized_text = vectorizer(X.values)
vocab_size = len(vectorizer.get_vocabulary())

class SimpleRNN(nn.Module):
    def __init__(self,vocab_size,embed_size,hidden_size,output_size):
        super(SimpleRNN,self).__init__()
        self.embedding = nn.Embedding(vocab_size,embedding_dim=embed_size,padding_idx=0)
        self.rnn = nn.RNN(embed_size,hidden_size,batch_first=True)
        self.dropout = nn.Dropout(0.5)
        self.fc = nn.Linear(hidden_size,output_size)
        
    def forward(self,X):
        X = self.embedding(X)
        out,ht =self.rnn(X)
        out = ht[-1]
        out = self.dropout(out)
        out = self.fc(out)
        return out

X_train, X_val, y_train, y_val = train_test_split(
    vectorized_text.numpy(),
    df_train['toxicity_ind'].values,
    test_size=0.2,
    random_state=42,
    stratify=df_train['toxicity_ind']
)

X_train = torch.tensor(X_train, dtype=torch.long)
X_val = torch.tensor(X_val, dtype=torch.long)

y_train = torch.tensor(y_train, dtype=torch.float32).reshape(-1,1)
y_val = torch.tensor(y_val, dtype=torch.float32).reshape(-1,1)

train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(
    train_dataset,
    batch_size=128,
    shuffle=True
)

val_dataset = TensorDataset(X_val, y_val)
val_loader = DataLoader(
    val_dataset,
    batch_size=128
)

model = SimpleRNN(vocab_size, embed_size=64, hidden_size=128, output_size=1)

positive_count = sum(df_train['toxicity_ind'])
negative_count = len(df_train) - positive_count
pos_weight = torch.tensor([negative_count / positive_count])

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = optim.Adam(model.parameters(), lr=0.001,weight_decay=1e-5)
best_val_loss = float('inf')
patience = 3
counter = 0

for epoch in range(10):
    model.train()
    epoch_loss = 0
    for batch_X, batch_y in train_loader:
        output = model(batch_X)
        loss = criterion(output, batch_y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    avg_loss = epoch_loss / len(train_loader)
    #print(f"Epoch {epoch+1}, Average Loss: {avg_loss:.4f}")

    val_loss = 0
    all_preds = []
    all_labels = []
    model.eval()
    with torch.no_grad():
        for batch_X, batch_y in val_loader:
            logits = model(batch_X)
            loss = criterion(logits, batch_y)
            val_loss += loss.item()

            probs = torch.sigmoid(logits)
            #threshold = 0.3
            #preds = (probs > threshold).float()
            all_preds.extend(probs.cpu().numpy())
            all_labels.extend(batch_y.cpu().numpy())

    avg_val_loss = val_loss / len(val_loader)
    all_preds = np.array(all_preds).flatten()
    all_labels = np.array(all_labels).flatten()

    print(
            f"Epoch {epoch+1} "
            f"Train Loss:{avg_loss:.4f} "
            f"Val Loss:{avg_val_loss:.4f}"
        )

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        counter = 0
        torch.save(
            model.state_dict(),
            "best_simple_rnn_model.pth"
        )
    else:
        counter += 1
        if counter >= patience:
            print("Early stopping")
            break

    for threshold in [0.3, 0.4, 0.5, 0.6]:
        preds = (all_preds > threshold).astype(int)
        print(f"\nThreshold = {threshold}")
        print(
            classification_report(
                all_labels,
                preds,
                digits=4
            )
        )
        print(
            "Confusion Matrix:\n",
            confusion_matrix(all_labels, preds)
        )

Epoch 1 Train Loss:1.2479 Val Loss:1.2433

Threshold = 0.3


c:\Users\nisha\OneDrive\Desktop\Nisha\GUVI\MiniProject\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\nisha\OneDrive\Desktop\Nisha\GUVI\MiniProject\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\nisha\OneDrive\Desktop\Nisha\GUVI\MiniProject\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavio

              precision    recall  f1-score   support

         0.0     0.0000    0.0000    0.0000     28670
         1.0     0.1017    1.0000    0.1846      3245

    accuracy                         0.1017     31915
   macro avg     0.0508    0.5000    0.0923     31915
weighted avg     0.0103    0.1017    0.0188     31915

Confusion Matrix:
 [[    0 28670]
 [    0  3245]]

Threshold = 0.4
              precision    recall  f1-score   support

         0.0     1.0000    0.0000    0.0001     28670
         1.0     0.1017    1.0000    0.1846      3245

    accuracy                         0.1017     31915
   macro avg     0.5508    0.5000    0.0923     31915
weighted avg     0.9087    0.1017    0.0188     31915

Confusion Matrix:
 [[    1 28669]
 [    0  3245]]

Threshold = 0.5
              precision    recall  f1-score   support

         0.0     0.8990    0.8062    0.8501     28670
         1.0     0.1046    0.2000    0.1373      3245

    accuracy                         0.7445     

c:\Users\nisha\OneDrive\Desktop\Nisha\GUVI\MiniProject\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\nisha\OneDrive\Desktop\Nisha\GUVI\MiniProject\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\nisha\OneDrive\Desktop\Nisha\GUVI\MiniProject\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavio

Epoch 2 Train Loss:1.2520 Val Loss:1.2403

Threshold = 0.3


...

Confusion Matrix:
 [[    0 28670]
 [    0  3245]]

Threshold = 0.4
              precision    recall  f1-score   support

         0.0     0.9229    0.5499    0.6892     28670
         1.0     0.1299    0.5938    0.2132      3245

    accuracy                         0.5544     31915
   macro avg     0.5264    0.5719    0.4512     31915
weighted avg     0.8422    0.5544    0.6408     31915

Confusion Matrix:
 [[15767 12903]
 [ 1318  1927]]

Threshold = 0.5
              precision    recall  f1-score   support

         0.0     0.9139    0.8160    0.8622     28670
         1.0     0.1648    0.3208    0.2177      3245

    accuracy                         0.7656     31915
   macro avg     0.5393    0.5684    0.5400     31915
weighted avg     0.8377    0.7656    0.7966     31915

Confusion Matrix:
 [[23394  5276]
 [ 2204  1041]]

Threshold = 0.6
              precision    recall  f1-score   support

         0.0     0.9138    0.8184    0.8635     28670
         1.0     0.1653    0.3177   

c:\Users\nisha\OneDrive\Desktop\Nisha\GUVI\MiniProject\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\nisha\OneDrive\Desktop\Nisha\GUVI\MiniProject\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\nisha\OneDrive\Desktop\Nisha\GUVI\MiniProject\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavio

Confusion Matrix:
 [[    0 28670]
 [    0  3245]]

Threshold = 0.4
              precision    recall  f1-score   support

         0.0     0.9753    0.0151    0.0298     28670
         1.0     0.1028    0.9966    0.1863      3245

    accuracy                         0.1149     31915
   macro avg     0.5390    0.5059    0.1081     31915
weighted avg     0.8866    0.1149    0.0457     31915

Confusion Matrix:
 [[  434 28236]
 [   11  3234]]

Threshold = 0.5
              precision    recall  f1-score   support

         0.0     0.9250    0.5416    0.6832     28670
         1.0     0.1313    0.6120    0.2162      3245

    accuracy                         0.5488     31915
   macro avg     0.5281    0.5768    0.4497     31915
weighted avg     0.8443    0.5488    0.6357     31915

Confusion Matrix:
 [[15528 13142]
 [ 1259  1986]]

Threshold = 0.6
              precision    recall  f1-score   support

         0.0     0.8991    0.9990    0.9464     28670
         1.0     0.5082    0.0096   

c:\Users\nisha\OneDrive\Desktop\Nisha\GUVI\MiniProject\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\nisha\OneDrive\Desktop\Nisha\GUVI\MiniProject\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\nisha\OneDrive\Desktop\Nisha\GUVI\MiniProject\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavio

Confusion Matrix:
 [[    0 28670]
 [    0  3245]]

Threshold = 0.4
              precision    recall  f1-score   support

         0.0     0.9761    0.0285    0.0554     28670
         1.0     0.1038    0.9938    0.1879      3245

    accuracy                         0.1266     31915
   macro avg     0.5399    0.5112    0.1216     31915
weighted avg     0.8874    0.1266    0.0689     31915

Confusion Matrix:
 [[  817 27853]
 [   20  3225]]

Threshold = 0.5
              precision    recall  f1-score   support

         0.0     0.9260    0.5243    0.6695     28670
         1.0     0.1303    0.6296    0.2159      3245

    accuracy                         0.5350     31915
   macro avg     0.5281    0.5770    0.4427     31915
weighted avg     0.8451    0.5350    0.6234     31915

Confusion Matrix:
 [[15033 13637]
 [ 1202  2043]]

Threshold = 0.6
              precision    recall  f1-score   support

         0.0     0.8991    0.9991    0.9465     28670
         1.0     0.5345    0.0096   